# Automatic Differentiation

In that section, deriving the gradient $\nabla_{\mathbf{w}} L = \mathbf{X}^\top (2\mathbf{r}) = (2, 4)$ of our two-parameter line model by hand was straightforward, but doing so across dozens of layers and millions of parameters would be tedious and error-prone. Modern deep learning frameworks automate this via *automatic differentiation* (*autograd*): as the forward pass evaluates the loss, the framework records the sequence of elementary operations and their inputs in a *computational graph*, and then traverses that graph backward (*backpropagation*) to apply the chain rule automatically at a small constant multiple of the forward cost.

In [ ]:
import tensorflow as tf

## Computing Gradients

### The Forward Pass

We first set up the line model $L(\mathbf{w}) = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2$ at $\mathbf{w} = (1, 0)$ from that section using framework tensors, writing only the forward pass:

In [ ]:
X = tf.constant([[1.0, 0.0], [1.0, 1.0], [1.0, 2.0]])  # [1, input]
y = tf.constant([1.0, 1.0, 0.0])
w = tf.constant([1.0, 0.0])

def loss(w):
    return tf.reduce_sum((tf.linalg.matvec(X, w) - y) ** 2)

<!-- d2l:prose id=autograd-md-3 fw=mxnet, pytorch, tensorflow -->

Because recording operations and saving intermediate activations consumes memory, frameworks track only operations depending on tensors explicitly marked for gradients.
A `tf.GradientTape` records operations executed inside its `with` block; wrapping `w` in `tf.Variable` causes the tape to track it automatically.

In [ ]:
w = tf.Variable(w)

In [ ]:
with tf.GradientTape() as t:  # record the operations on the tape t
    L = loss(w)
L

The recorded trace forms the *computational graph* shown in the figure, breaking the loss into four primitive operations: $\hat{\mathbf{y}} = \mathbf{X}\mathbf{w}$, $\mathbf{r} = \hat{\mathbf{y}} - \mathbf{y}$, $\mathbf{s} = \mathbf{r} \odot \mathbf{r}$, and $L = \sum_i s_i$.

![The computational graph of $L = \|\mathbf{X}\mathbf{w} - \mathbf{y}\|^2$ at $\mathbf{w} = (1, 0)$, with one node for $\mathbf{w}$ and for each value computed from it; the constants $\mathbf{X}$ and $\mathbf{y}$ are not drawn, and $\mathbf{r} \odot \mathbf{r}$ is the elementwise square. Each node shows its value (black) and the gradient of $L$ with respect to it (red). The forward pass (blue) runs from left to right; the backward pass (red) starts from $1$ at $L$ and applies the local rule of each operation, from right to left.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-comp-graph.svg)

### The Backward Pass

Triggering the backward pass propagates sensitivities from $\partial L/\partial L = 1$ at the output back to $\mathbf{w}$ through each node's local derivative rule ($\nabla_{\mathbf{s}} L = \mathbf{1} \to \nabla_{\mathbf{r}} L = 2\mathbf{r} \to \nabla_{\hat{\mathbf{y}}} L = 2\mathbf{r} \to \nabla_{\mathbf{w}} L = \mathbf{X}^\top(2\mathbf{r})$), reproducing $(2, 4)$:

`t.gradient(L, w)` computes the gradient of `L` with respect to `w` from the tape `t`.

In [ ]:
t.gradient(L, w)

Because local backward rules require intermediate activations from the forward pass (such as $\mathbf{r}$ and $\mathbf{X}$), training consumes substantially more memory than inference.

### Gradient Accumulation

What happens when we run a second backward pass at the same parameter vector $\mathbf{w}$?

In [ ]:
with tf.GradientTape() as t:
    L = loss(w)
t.gradient(L, w)  # a new tensor; nothing is stored in w

`t.gradient(L, w)` returns a fresh tensor without mutating `w`, giving $(2, 4)$ again.

### Vector-Valued Functions

When an operation produces a vector of $m$ outputs $\mathbf{u} \in \mathbb{R}^m$ from $n$ inputs $\mathbf{x} \in \mathbb{R}^n$ with $m \times n$ Jacobian $\mathbf{J}$, a backward pass seeded with a weight vector $\mathbf{v} \in \mathbb{R}^m$ computes the transposed *vector--Jacobian product* $\mathbf{J}^\top \mathbf{v} = \nabla_{\mathbf{x}}(\mathbf{v}^\top \mathbf{u})$ without materializing $\mathbf{J}$. Passing $\mathbf{v} = \mathbf{1}$ on the per-example squared errors $(r_1^2, r_2^2, r_3^2)$ differentiates their sum:

`t.gradient` accepts $\mathbf{v}$ via `output_gradients` (defaulting to $\mathbf{1}$ if omitted).

In [ ]:
with tf.GradientTape() as t:
    losses = (tf.linalg.matvec(X, w) - y) ** 2  # per-example losses
t.gradient(losses, w, output_gradients=tf.ones(3))  # v = 1

## Controlling the Graph

### Detaching Values

*Detaching* a tensor creates a view or copy with the same numerical values but no link to its ancestors in the computational graph, so backpropagation treats it as a constant. Consider $u = x^2$ and $z = u x$ on $\mathbf{x} = (0, 1, 2, 3)$ (the figure): when $u$ is attached, gradient flows along both paths to give $\partial z/\partial x = 3x^2$; when $u$ is detached to $c$, differentiating $z = c x$ treats $c$ as a constant and gives $\partial z/\partial x = c = x^2$ (essential whenever a network computes its own bootstrap target, as in Q-learning, that section).

![The product $z = ux$ with $u = x^2$, (a) attached and (b) detached. Blue arrows are operations of the forward pass; red arrows show the backward pass, labeled with the local derivatives it multiplies by. Detaching $u$ creates a constant $c$ with the same value and no connection to $x$.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-detach.svg)

`tf.stop_gradient(u)` passes `u` forward while blocking backward gradients.

In [ ]:
x = tf.Variable(tf.range(4.0))
with tf.GradientTape() as t:
    u = x * x
    c = tf.stop_gradient(u)  # values of u, no history
    z = c * x
t.gradient(z, x)

In [ ]:
with tf.GradientTape() as t:
    u = x * x
    z = u * x
t.gradient(z, x)

### Disabling Gradient Tracking

During validation, inference, or parameter updates $\mathbf{w} \leftarrow \mathbf{w} - \eta \nabla L(\mathbf{w})$, we disable gradient tracking so those operations are not recorded on the loss graph:

Code outside `tf.GradientTape` runs unrecorded; `w.assign_sub(...)` updates `w` in place.

In [ ]:
with tf.GradientTape() as t:
    L = loss(w)
w.assign_sub(0.1 * t.gradient(L, w))  # outside the tape
w.numpy(), round(float(loss(w)), 4)

As in that section, one step with $\eta = 0.1$ moves $\mathbf{w}$ from $(1, 0)$ to $(0.8, -0.4)$ and lowers the loss from $1$ to $0.4$.

## Differentiating Programs

### Control Flow

Because autograd records the sequence of elementary operations that *actually executed* on a given input, it differentiates Python functions containing data-dependent `while` loops and `if`/`else` branches directly:

In [ ]:
def f(a):
    b = a * 2
    while tf.norm(b) < 1000:
        b = b * 2
    if tf.reduce_sum(b) > 0:
        c = b
    else:
        c = 100 * b
    return c

For scalar $a$, `f(a) = k * a` is piecewise linear with slope $k = f(a)/a$ ($k = 2^{12} = 4096$ at $a = 0.3$; $k = 100 \cdot 2^{11} = 204{,}800$ at $a = -0.75$, the figure), which autograd recovers exactly:

In [ ]:
for value in (0.3, -0.75):
    a = tf.Variable(value)
    with tf.GradientTape() as t:
        fa = f(a)
    print(f'a = {value:5.2f}   f(a) / a = {float(fa) / value:6.0f}   '
          f'gradient = {float(t.gradient(fa, a)):6.0f}')

![The graphs recorded by `f` for two inputs. For $a = 0.3$, `f` doubles $a$ $12$ times and the first branch runs; for $a = -0.75$, it doubles $a$ $11$ times and the second branch runs. Each graph is linear in $a$, with its own slope.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-dynamic.svg)

Keep in mind that autograd differentiates the *executed trace* rather than the abstract mathematical function: at a threshold point where the loop count jumps, $f(a)$ is discontinuous, yet autograd returns the slope of whichever branch ran.

## Forward and Reverse Mode

For a mapping $\mathbf{f} : \mathbb{R}^n \to \mathbb{R}^m$ with $m \times n$ Jacobian $\mathbf{J}$, the chain rule can be traversed in either direction (the figure; see @Wengert.1964; @Linnainmaa.1970 and that section):

* **Reverse mode (vector--Jacobian product, VJP):** A reverse sweep seeded with $\mathbf{v} \in \mathbb{R}^m$ at the outputs computes $\mathbf{J}^\top \mathbf{v}$, extracting one *row* of $\mathbf{J}$ per sweep ($m$ sweeps for the full Jacobian).
* **Forward mode (Jacobian--vector product, JVP):** A forward sweep seeded with a tangent $\mathbf{d} \in \mathbb{R}^n$ at the inputs propagates derivatives forward alongside values to compute $\mathbf{J}\mathbf{d}$, extracting one *column* of $\mathbf{J}$ per sweep ($n$ sweeps for the full Jacobian, with constant $2\times$ memory overhead).

![Two ways to fill the $m \times n$ Jacobian $\mathbf{J}$ of a function with $n$ inputs and $m$ outputs: (a) forward mode, $n$ sweeps in all; (b) reverse mode, $m$ sweeps in all.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/autograd-fwd-vs-rev.svg)

TensorFlow provides `GradientTape.jacobian` (reverse mode) and `tf.autodiff.ForwardAccumulator` (forward mode).

A scalar loss function has $m = 1$ output and $n$ parameters, so its Jacobian is a single row $\nabla_{\mathbf{w}} L^\top$. Reverse mode (*backpropagation*) computes all $n$ components of the gradient in *one sweep*, whereas forward mode would require $n$ sweeps—explaining why reverse mode is universal for training deep networks.

## Summary

* Automatic differentiation records the forward pass on a computational graph and applies the chain rule backward to compute exact gradients (reproducing $\nabla_{\mathbf{w}} L = (2, 4)$ on our line model).
* Detaching an intermediate tensor severs its graph history so backpropagation treats it as a constant, while disabling gradient tracking keeps parameter updates and inference evaluations off the graph.
* Because the graph records the operations that actually ran, autograd differentiates programs with dynamic loops and branches.
* One forward sweep computes one column of an $m \times n$ Jacobian ($\mathbf{J}\mathbf{d}$), whereas one reverse sweep computes one row ($\mathbf{J}^\top\mathbf{v}$); for a scalar loss ($m = 1$), a single reverse sweep yields the entire gradient.

## Exercises

1. [code] **Detached intermediates.** Let `u = x * x` and
   `c = tf.stop_gradient(u)` inside a persistent `tf.GradientTape` `t`.
   Before running any code, predict `t.gradient(z, x)` for each of
   `z = c * c`, `z = u * c`, and `z = u * u`, all computed inside `t`. Check
   your predictions, and explain the ratio between the results for
   `z = u * c` and `z = u * u`.

2. [code] **Manual parameter updates.** Perform the gradient step of this
   section as `w = w - 0.1 * g`, where `g` is the gradient returned by the
   tape, instead of with `assign_sub`, and inspect `type(w)`. Predict what
   `gradient` returns for a second forward pass under a new tape, check, and
   explain why a training loop updates variables with `assign_sub`.

3. [code] **From scalar to vector input.** Replace the scalar input `a` of
   the function `f` defined in that section by a
   random vector of length 5, and differentiate the sum of the entries of
   `fa = f(a)` with respect to `a`.
    1. Express `f(a)` in terms of `a`, and give the Jacobian of `f` at `a`
       and the gradient of the sum of the entries of `fa`.
    1. Compute this gradient with automatic differentiation and check
       whether it still equals `fa / a` elementwise.
    1. Describe the set of inputs `a` at which `f` jumps. Which part of
       this set has no counterpart for a scalar input?

1. [code] **Programs versus functions.** Automatic differentiation
   differentiates the operations that a program executes, not the function
   that the program computes.
    1. The functions $|x|$ and $\sqrt{x^2}$ are equal. Compute the
       derivative of each at $x = 0$ by automatic differentiation, using the
       framework's absolute value and square root. Explain both results from
       the local derivatives of the operations involved. Neither function
       has a derivative at $0$; which of the two results does more harm in
       training, and why?
    1. Write a program that returns $x^2$ for every input $x$ but whose
       automatic derivative at $x = 0$ is $1$ instead of $0$. Explain why
       automatic differentiation cannot detect the difference.

1. **Cost of second derivatives.** ● The counting argument in
   that section puts the cost of the gradient of a
   scalar loss with $n$ parameters at one reverse sweep. Use the same argument to
   estimate, in units of function evaluations, the cost of the full Hessian
   (the $n \times n$ matrix of second derivatives) and the cost of a single
   Hessian–vector product. Explain the difference between the two costs.

[Discussions](https://d2l.discourse.group/t/200)